# 02 -- Model Training: Bayesian Marketing Mix Model

This notebook walks through the full training pipeline for the Bayesian MMM
using **PyMC-Marketing**. It mirrors `src/train.py` but adds inline
visualisations and diagnostics.

**Key results from the training run:**

| Metric | Value |
|--------|-------|
| Sampler | pymc, 4 chains x 1000 draws |
| Wall time | ~11 seconds |
| Max R-hat | 1.0067 (PASSED) |
| Min ESS bulk | 2032 (PASSED) |
| Min ESS tail | 1384 (PASSED) |
| Out-of-sample MAPE | 3.9% (target < 15%) |
| Out-of-sample R-squared | 0.9145 |
| Divergences | 7 (acceptable) |
| Parameter recovery | 9/10 within 95% HDI |

## 1. Setup and Data Loading

In [ ]:
import sys
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

# Module resolution fix so we can import from src/
sys.path.insert(0, os.path.abspath(".."))

from src.feature_engineering import (
    CHANNEL_COLUMNS,
    CONTROL_COLUMNS,
    TARGET_COLUMN,
    add_features,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print(f"Channel columns: {CHANNEL_COLUMNS}")
print(f"Control columns: {CONTROL_COLUMNS}")
print(f"Target column:   {TARGET_COLUMN}")

In [ ]:
# Load the synthetic weekly dataset and add engineered features
raw_path = os.path.join("..", "data", "synthetic", "mmm_weekly_data.csv")
df = pd.read_csv(raw_path, parse_dates=["date_week"])
df = add_features(df)

print(f"Shape: {df.shape}")
print(f"Date range: {df.date_week.min().date()} to {df.date_week.max().date()}")
df.head()

## 2. Train / Test Split

We use a chronological split: the first **156 weeks** (~3 years) for training
and the remaining **52 weeks** (~1 year) for out-of-sample evaluation.
This mirrors a realistic deployment scenario where we train on historical data
and validate on the most recent year.

In [ ]:
TRAIN_WEEKS = 156

train_df = df.iloc[:TRAIN_WEEKS].copy().reset_index(drop=True)
test_df = df.iloc[TRAIN_WEEKS:].copy().reset_index(drop=True)

X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]
X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

print(f"Train: {len(train_df)} weeks  ({X_train.date_week.min().date()} to {X_train.date_week.max().date()})")
print(f"Test:  {len(test_df)} weeks  ({X_test.date_week.min().date()} to {X_test.date_week.max().date()})")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

split_date = df.iloc[TRAIN_WEEKS]["date_week"]

ax.plot(train_df["date_week"], y_train, color="#2980b9", linewidth=1.2, label="Train (156 weeks)")
ax.plot(test_df["date_week"], y_test, color="#e67e22", linewidth=1.2, label="Test (52 weeks)")
ax.axvline(x=split_date, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Split Point")

ax.set_title("Weekly Revenue: Train / Test Split", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (EUR)")
ax.legend(loc="upper left")
ax.ticklabel_format(style="plain", axis="y")
plt.tight_layout()
plt.show()

## 3. MMM Configuration

We use the `MMM` class from `pymc_marketing.mmm` with the following components:

- **GeometricAdstock(l_max=8)** -- models carry-over effects up to 8 weeks.
  Each channel learns its own decay rate (alpha). Higher alpha means longer
  carry-over (e.g., TV is expected to have a higher alpha than search).

- **LogisticSaturation()** -- models diminishing returns. The saturation
  lambda parameter controls how quickly a channel saturates. Lower lambda
  means the channel saturates faster.

- **yearly_seasonality=2** -- two Fourier pairs to capture annual patterns
  in revenue not explained by media spend.

- **Control variables** -- `competitor_index` (competitive pressure) and
  `event_flag` (promotional events) plus a normalised linear trend `t`.

In [ ]:
from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation, MMM

L_MAX = 8
YEARLY_SEASONALITY = 2

mmm = MMM(
    date_column="date_week",
    channel_columns=CHANNEL_COLUMNS,
    control_columns=CONTROL_COLUMNS,
    adstock=GeometricAdstock(l_max=L_MAX),
    saturation=LogisticSaturation(),
    yearly_seasonality=YEARLY_SEASONALITY,
)

print("MMM configured.")
print(f"  Channels:     {CHANNEL_COLUMNS}")
print(f"  Controls:     {CONTROL_COLUMNS}")
print(f"  Adstock:      GeometricAdstock(l_max={L_MAX})")
print(f"  Saturation:   LogisticSaturation()")
print(f"  Seasonality:  yearly_seasonality={YEARLY_SEASONALITY}")

## 4. Model Fitting

The model is fitted with 4 MCMC chains, each drawing 1000 posterior samples
after 1000 tuning steps. On this synthetic dataset the fit completes in
roughly 11 seconds using the default PyMC sampler.

**If you want to re-fit from scratch**, uncomment the `mmm.fit(...)` call
below. Otherwise, load the pre-fitted trace from disk to skip the sampling
step.

In [ ]:
# ---- Option A: Re-fit the model (takes ~12 seconds) ----
# import time
# t0 = time.time()
# mmm.fit(X=X_train, y=y_train, random_seed=42)
# elapsed = time.time() - t0
# print(f"Fitting complete in {elapsed:.1f} seconds")

# ---- Option B: Load the pre-fitted trace ----
trace_path = os.path.join("..", "models", "mmm_trace", "trace.nc")
idata = az.from_netcdf(trace_path)
print(f"Loaded pre-fitted trace from {trace_path}")
print(f"Posterior shape: {dict(idata.posterior.dims)}")

## 5. Convergence Diagnostics

Good MCMC convergence requires:

- **R-hat < 1.01** for all parameters (chains have mixed well).
- **ESS bulk > 400** (enough effective samples for posterior means).
- **ESS tail > 400** (enough effective samples for credible intervals).

We also inspect trace plots visually for the adstock alpha and saturation
lambda parameters to confirm the chains are stationary and well-mixed.

In [ ]:
# If we re-fitted, use mmm.idata; otherwise use the loaded idata
if hasattr(mmm, "idata") and mmm.idata is not None:
    trace = mmm.idata
else:
    trace = idata

# Summary of key parameters
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    summary = az.summary(trace, round_to=4)

# Filter to adstock and saturation parameters for readability
key_params = summary[
    summary.index.str.contains("alpha|lam|saturation|adstock", case=False)
]

print("Key Parameter Summary")
print("=" * 80)
key_params

In [ ]:
# Overall convergence check
valid = summary.dropna(subset=["r_hat"])
max_rhat = float(valid["r_hat"].max()) if len(valid) > 0 else 1.0
min_ess_bulk = float(valid["ess_bulk"].min()) if len(valid) > 0 else 0.0
min_ess_tail = float(valid["ess_tail"].min()) if len(valid) > 0 else 0.0

rhat_ok = max_rhat < 1.01
ess_bulk_ok = min_ess_bulk > 400
ess_tail_ok = min_ess_tail > 400

print(f"Max R-hat:       {max_rhat:.4f}  {'PASSED' if rhat_ok else 'FAILED'}  (target < 1.01)")
print(f"Min ESS (bulk):  {min_ess_bulk:.0f}    {'PASSED' if ess_bulk_ok else 'FAILED'}  (target > 400)")
print(f"Min ESS (tail):  {min_ess_tail:.0f}    {'PASSED' if ess_tail_ok else 'FAILED'}  (target > 400)")

In [ ]:
# Trace plots for adstock alpha parameters
adstock_vars = [v for v in trace.posterior.data_vars if "alpha" in v.lower() or "adstock" in v.lower()]
if adstock_vars:
    az.plot_trace(trace, var_names=adstock_vars, figsize=(14, 3 * len(adstock_vars)))
    plt.suptitle("Trace Plots: Adstock Alpha Parameters", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No adstock alpha variables found in trace.")

In [ ]:
# Trace plots for saturation lambda parameters
sat_vars = [v for v in trace.posterior.data_vars if "lam" in v.lower() or "saturation" in v.lower()]
if sat_vars:
    az.plot_trace(trace, var_names=sat_vars, figsize=(14, 3 * len(sat_vars)))
    plt.suptitle("Trace Plots: Saturation Lambda Parameters", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No saturation lambda variables found in trace.")

## 6. Out-of-Sample Predictions

We evaluate the model on the held-out 52 test weeks. Because re-running
`mmm.predict()` requires a fully fitted model object, we provide two paths:

1. If the MMM was fitted in this session, call `mmm.predict(X=X_test)`.
2. Otherwise, load the saved metadata which contains the aggregate metrics,
   and use the decomposition data to reconstruct predictions.

Target performance: **MAPE < 15%**.

In [ ]:
# Load metadata with pre-computed out-of-sample metrics
meta_path = os.path.join("..", "models", "metadata.json")
with open(meta_path) as f:
    metadata = json.load(f)

oos = metadata["out_of_sample"]
mae = oos["mae"]
mape = oos["mape"]
r2 = oos["r2"]

print("Out-of-Sample Performance (52 test weeks)")
print("=" * 45)
print(f"  MAE:  {mae:,.0f} EUR")
print(f"  MAPE: {mape:.1f}%")
print(f"  R2:   {r2:.4f}")
print()
if mape < 15:
    print(f"MAPE {mape:.1f}% is well below the 15% target -- PASSED.")
else:
    print(f"MAPE {mape:.1f}% exceeds the 15% target -- needs investigation.")

In [ ]:
# If the MMM was fitted this session, generate actual vs predicted plot
fitted_this_session = hasattr(mmm, "idata") and mmm.idata is not None

if fitted_this_session:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        posterior_pred = mmm.predict(X=X_test)

    # Extract mean predictions
    try:
        if hasattr(posterior_pred, "posterior_predictive"):
            pp = posterior_pred.posterior_predictive
            var_name = list(pp.data_vars)[0]
            y_pred = pp[var_name].mean(dim=["chain", "draw"]).values.flatten()
        elif isinstance(posterior_pred, np.ndarray):
            y_pred = posterior_pred.flatten()
        else:
            y_pred = np.array(posterior_pred).flatten()
    except Exception:
        pp = mmm.idata.posterior_predictive
        var_name = list(pp.data_vars)[0]
        y_pred = pp[var_name].mean(dim=["chain", "draw"]).values.flatten()

    y_actual = y_test.values
    min_len = min(len(y_pred), len(y_actual))
    y_pred = y_pred[:min_len]
    y_actual = y_actual[:min_len]

    fig, axes = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={"height_ratios": [3, 1]})

    # Top panel: actual vs predicted
    ax = axes[0]
    test_dates = X_test["date_week"].values[:min_len]
    ax.plot(test_dates, y_actual, color="#2c3e50", linewidth=1.5, label="Actual")
    ax.plot(test_dates, y_pred, color="#e74c3c", linewidth=1.5, linestyle="--", label="Predicted")
    ax.fill_between(test_dates, y_actual, y_pred, alpha=0.15, color="#e74c3c")
    ax.set_title("Out-of-Sample: Actual vs Predicted Revenue", fontsize=14, fontweight="bold")
    ax.set_ylabel("Revenue (EUR)")
    ax.legend(loc="upper left")
    ax.ticklabel_format(style="plain", axis="y")

    # Bottom panel: residuals
    ax2 = axes[1]
    residuals = y_actual - y_pred
    ax2.bar(test_dates, residuals, color="#95a5a6", width=5)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_title("Residuals (Actual - Predicted)", fontsize=12)
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Residual (EUR)")
    ax2.ticklabel_format(style="plain", axis="y")

    plt.tight_layout()
    plt.show()
else:
    print("Model was not fitted in this session.")
    print("Using pre-computed metrics from metadata.json (shown above).")
    print("To generate the actual-vs-predicted plot, uncomment the mmm.fit() call in Section 4.")

## 7. Parameter Recovery

Since we trained on synthetic data with known ground-truth parameters,
we can evaluate whether the Bayesian model recovers the true values.
A parameter is considered "recovered" if its true value falls within
the 95% highest density interval (HDI) of the posterior.

**Result: 9 out of 10 parameters recovered.**
The only miss was the TV adstock alpha, whose true value (0.70) fell
outside the posterior HDI.

In [ ]:
# Load ground-truth parameters
params_path = os.path.join("..", "data", "synthetic", "true_params.json")
with open(params_path) as f:
    true_params = json.load(f)

# Load recovery results from metadata
recovery = metadata["parameter_recovery"]

# Display adstock alpha recovery
print("Adstock Alpha Recovery")
print("=" * 75)
print(f"{'Channel':<12} {'True':>8} {'Estimated':>10} {'HDI 2.5%':>10} {'HDI 97.5%':>10} {'Status':>8}")
print("-" * 75)
for ch, vals in recovery["adstock_alphas"].items():
    status = "OK" if vals["in_hdi"] else "MISS"
    print(f"{ch:<12} {vals['true']:>8.2f} {vals['estimated']:>10.4f} {vals['hdi_3']:>10.4f} {vals['hdi_97']:>10.4f} {status:>8}")

print()
print("Saturation Lambda Recovery")
print("=" * 75)
print(f"{'Channel':<12} {'True':>8} {'Estimated':>10} {'HDI 2.5%':>10} {'HDI 97.5%':>10} {'Status':>8}")
print("-" * 75)
for ch, vals in recovery["saturation_lambdas"].items():
    status = "OK" if vals["in_hdi"] else "MISS"
    print(f"{ch:<12} {vals['true']:>8.2f} {vals['estimated']:>10.4f} {vals['hdi_3']:>10.4f} {vals['hdi_97']:>10.4f} {status:>8}")

# Count recoveries
n_ok = sum(1 for v in recovery["adstock_alphas"].values() if v["in_hdi"])
n_ok += sum(1 for v in recovery["saturation_lambdas"].values() if v["in_hdi"])
n_total = len(recovery["adstock_alphas"]) + len(recovery["saturation_lambdas"])
print(f"\nRecovered: {n_ok}/{n_total} parameters within 95% HDI")

In [ ]:
# Bar chart: True vs Estimated parameter values
channels = list(recovery["adstock_alphas"].keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Adstock Alphas ---
ax = axes[0]
x = np.arange(len(channels))
width = 0.35

true_alphas = [recovery["adstock_alphas"][ch]["true"] for ch in channels]
est_alphas = [recovery["adstock_alphas"][ch]["estimated"] for ch in channels]
hdi_lo = [recovery["adstock_alphas"][ch]["hdi_3"] for ch in channels]
hdi_hi = [recovery["adstock_alphas"][ch]["hdi_97"] for ch in channels]
in_hdi = [recovery["adstock_alphas"][ch]["in_hdi"] for ch in channels]

bars_true = ax.bar(x - width / 2, true_alphas, width, label="True", color="#2c3e50", alpha=0.8)
bars_est = ax.bar(x + width / 2, est_alphas, width, label="Estimated", color="#3498db", alpha=0.8)

# Error bars for HDI
yerr_lo = [est_alphas[i] - hdi_lo[i] for i in range(len(channels))]
yerr_hi = [hdi_hi[i] - est_alphas[i] for i in range(len(channels))]
ax.errorbar(x + width / 2, est_alphas, yerr=[yerr_lo, yerr_hi],
            fmt="none", ecolor="black", capsize=4, linewidth=1.2)

# Mark misses with a red border
for i, ok in enumerate(in_hdi):
    if not ok:
        bars_true[i].set_edgecolor("red")
        bars_true[i].set_linewidth(2)
        ax.annotate("MISS", xy=(x[i] - width / 2, true_alphas[i]),
                    xytext=(x[i] - width / 2, true_alphas[i] + 0.08),
                    ha="center", fontsize=9, color="red", fontweight="bold")

ax.set_title("Adstock Alpha: True vs Estimated", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([ch.upper() for ch in channels])
ax.set_ylabel("Alpha")
ax.legend()
ax.set_ylim(0, 1.0)

# --- Saturation Lambdas ---
ax = axes[1]

true_lams = [recovery["saturation_lambdas"][ch]["true"] for ch in channels]
est_lams = [recovery["saturation_lambdas"][ch]["estimated"] for ch in channels]
hdi_lo_l = [recovery["saturation_lambdas"][ch]["hdi_3"] for ch in channels]
hdi_hi_l = [recovery["saturation_lambdas"][ch]["hdi_97"] for ch in channels]
in_hdi_l = [recovery["saturation_lambdas"][ch]["in_hdi"] for ch in channels]

bars_true_l = ax.bar(x - width / 2, true_lams, width, label="True", color="#2c3e50", alpha=0.8)
bars_est_l = ax.bar(x + width / 2, est_lams, width, label="Estimated", color="#e67e22", alpha=0.8)

yerr_lo_l = [est_lams[i] - hdi_lo_l[i] for i in range(len(channels))]
yerr_hi_l = [hdi_hi_l[i] - est_lams[i] for i in range(len(channels))]
ax.errorbar(x + width / 2, est_lams, yerr=[yerr_lo_l, yerr_hi_l],
            fmt="none", ecolor="black", capsize=4, linewidth=1.2)

for i, ok in enumerate(in_hdi_l):
    if not ok:
        bars_true_l[i].set_edgecolor("red")
        bars_true_l[i].set_linewidth(2)
        ax.annotate("MISS", xy=(x[i] - width / 2, true_lams[i]),
                    xytext=(x[i] - width / 2, true_lams[i] + 0.3),
                    ha="center", fontsize=9, color="red", fontweight="bold")

ax.set_title("Saturation Lambda: True vs Estimated", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([ch.upper() for ch in channels])
ax.set_ylabel("Lambda")
ax.legend()

plt.tight_layout()
plt.show()

## 8. Channel Contributions

Channel contributions tell us how much revenue each media channel is
responsible for, after accounting for adstock carry-over and saturation.
The "base" component captures revenue that would occur without any
media spend (driven by trend, seasonality, and controls).

If the model was fitted this session, we call
`mmm.compute_channel_contribution_original_scale()`. Otherwise, we load
the pre-computed decomposition from the training artifacts.

In [ ]:
# Load pre-computed decomposition
decomp_path = os.path.join("..", "models", "precomputed", "decomposition.json")
with open(decomp_path) as f:
    decomp = json.load(f)

weeks_data = decomp["weeks"]
decomp_df = pd.DataFrame(weeks_data)
decomp_df["date_week"] = pd.to_datetime(decomp_df["date_week"])

channel_names = ["tv", "ooh", "print", "facebook", "search"]

print(f"Decomposition data: {len(decomp_df)} weeks")
print(f"\nTotal attribution (% of revenue):")
for ch in channel_names:
    pct = decomp["pct"].get(ch, 0)
    print(f"  {ch:>10}: {pct:.1f}%")
base_pct = 100 - sum(decomp["pct"].get(ch, 0) for ch in channel_names)
print(f"  {'base':>10}: {base_pct:.1f}%")

In [ ]:
# Stacked area chart of weekly channel contributions
channel_colors_map = {
    "tv": "#e74c3c",
    "ooh": "#3498db",
    "print": "#2ecc71",
    "facebook": "#9b59b6",
    "search": "#f39c12",
    "base": "#95a5a6",
}

fig, ax = plt.subplots(figsize=(14, 6))

stack_cols = ["base"] + channel_names
stack_data = np.array([decomp_df[col].values for col in stack_cols])
colors = [channel_colors_map[col] for col in stack_cols]
labels = [col.upper() if col != "base" else "Base" for col in stack_cols]

ax.stackplot(
    decomp_df["date_week"],
    stack_data,
    labels=labels,
    colors=colors,
    alpha=0.8,
)

# Overlay actual revenue line
ax.plot(decomp_df["date_week"], decomp_df["revenue_actual"],
        color="black", linewidth=1.2, linestyle="--", label="Actual Revenue")

ax.set_title("Weekly Revenue Decomposition by Channel", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (EUR)")
ax.legend(loc="upper left", ncol=4, fontsize=9)
ax.ticklabel_format(style="plain", axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart of total channel attribution
totals = decomp["totals"]
total_revenue = decomp_df["revenue_actual"].sum()

# Compute base total
channel_total = sum(totals.get(ch, 0) for ch in channel_names)
base_total = total_revenue - channel_total

pie_labels = ["Base"] + [ch.upper() for ch in channel_names]
pie_values = [base_total] + [totals.get(ch, 0) for ch in channel_names]
pie_colors = [channel_colors_map["base"]] + [channel_colors_map[ch] for ch in channel_names]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    pie_values,
    labels=pie_labels,
    colors=pie_colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
    textprops={"fontsize": 11},
)
for autotext in autotexts:
    autotext.set_fontsize(10)

ax.set_title("Total Revenue Attribution by Channel", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

## 9. Summary

### Model specification

A Bayesian Marketing Mix Model was fitted using PyMC-Marketing with
geometric adstock (l_max = 8) and logistic saturation transformations
on five paid media channels: TV, out-of-home, print, Facebook, and search.
Yearly seasonality was captured with two Fourier pairs, and competitor
index plus event flag served as control variables.

### Convergence

All chains converged. The maximum R-hat across all parameters was 1.0067
(well below the 1.01 threshold). Effective sample sizes were healthy:
ESS bulk minimum of 2,032 and ESS tail minimum of 1,384. Seven divergences
were observed, which is acceptable for this model complexity.

### Predictive accuracy

Out-of-sample performance on 52 held-out weeks was strong:
- MAPE of 3.9% (target was < 15%)
- R-squared of 0.9145

### Parameter recovery

9 out of 10 ground-truth parameters fell within the model's 95% HDI.
The only miss was the TV adstock alpha (true = 0.70), where the posterior
HDI was [0.00, 0.60]. This is a known difficulty -- adstock and saturation
parameters can trade off against each other, especially for the highest-spend
channel where saturation effects are strongest.

### Next steps

- Notebook `03_model_diagnostics.ipynb` will explore response curves,
  ROAS estimates, and budget optimisation.
- The pre-computed JSON artifacts in `models/precomputed/` power the
  FastAPI + React dashboard for interactive exploration.